# Batch-effect bifurcation -- trajectories only

Section 2 of `bifurcation_estimate` on its own: the estimates are LOADED from
`{TAG}_seed{k}_est.npz` (nothing is re-estimated; `G` is rebuilt deterministically from the seed,
so it is identical to what every baseline saw). Fits ours: UOT maps / flow OT-CFM unshared / flow
UOT-map unshared, scored in the 2-D signal plane AND the full ambient dimension, then saved.

In [ ]:
import os, sys, time, json
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
import uotreg as U
from uotreg import simulation as sim, trajectory_metrics as TM
from uotreg.metrics import w2
from uotreg.plotting import plot_trajectory_panel

## Parameters + loader + fitter
`K_DEFAULT` holds the trajectory knobs only -- the estimation knobs are baked into the saved npz.

In [ ]:
DIM        = globals().get("DIM", 10)
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE      = 1
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = globals().get("SAVE", 0)
DEVICE     = globals().get("DEVICE", "cpu")
STRENGTH   = 0.5
TRAJ_T     = list(range(1, 9))             # times 1..8 (split onset ~t=4)
# where the saved estimates are read from: at SMOKE = 0 the SHIPPED full-quality estimates;
# at SMOKE = 1 your own `new_results/` run of the estimate stage (the stages hand off through disk)
EST_ROOT   = P.results("batcheffectnew") if SMOKE else P.shipped("batcheffectnew")
RESULTS_W  = P.results("batcheffectnew", write=True)           # write (only when SAVE)
TAG        = f"bifurcation_d{DIM}_new" + ("_quick" if SMOKE else "")
METHODS    = ["ours: UOT maps",                  # composed UOT maps (the paper's trajectory method)
              "ours: flow OT-CFM unshared",      # flow, minibatch-OT coupling, field per interval
              "ours: flow UOT-map unshared"]     # flow trained against OUR maps (chain reused, fit once)
MK = [("adher_labeled", "adher*"), ("adher_indiv", "indiv"), ("w2_path", "w2_path"),
      ("w2_path_full", f"w2_{DIM}d"), ("spread_post", "spread"), ("roughness", "rough"),
      ("balanceA", "balA")]

K_DEFAULT = dict(traj_hidden=(64 if SMOKE else 256), map_layers=5,          # UOT maps
                 d_iters=(20 if SMOKE else 250), t_iters=(5 if SMOKE else 100),
                 map_batch=64,                                             # <- tunable (cells / map step)
                 flow_hidden=(64 if SMOKE else 128), flow_layers=4,        # flows (hidden 128)
                 flow_iters=(300 if SMOKE else 3000),                      # <- tunable
                 flow_batch=(64 if SMOKE else 256),                        # <- tunable
                 flow_lr=1e-3, flow_sigma=0.0, flow_n_per=20, flow_field="mlp", flow_seed=0,
                 tau=5.0, uot_seed=0)                                      # uot_seed=None = unseeded

EST, TRAJS, FULL, ROWS = {}, {}, {}, {}    # seed -> loaded estimate / 2-D trajs / full-dim trajs / rows

_est_path = lambda seed: os.path.join(EST_ROOT, f"{TAG}_seed{seed}_est.npz")


def load_seed(seed, viz=False):
    """Section-1: LOAD seed's saved estimate and rebuild everything Section 2 needs.
    `ours_series` is reconstructed exactly as `estimate_seed` built it: [raw[0]] + est[1:]."""
    if not os.path.exists(_est_path(seed)):
        raise FileNotFoundError(
            f"no saved estimate at {_est_path(seed)}.\n"
            "  At SMOKE = 1 the stages hand off through disk: run the estimate notebook first, "
            "with SAVE = 1 (its output lands in new_results/).\n"
            "  At SMOKE = 0 the shipped full-quality estimates are read.")
    z = np.load(_est_path(seed), allow_pickle=True)
    est = [np.asarray(z["est_series"][i], np.float32) for i in range(z["est_series"].shape[0])]
    raw = [np.asarray(z["raw_series"][i], np.float32) for i in range(z["raw_series"].shape[0])]
    n_per = raw[0].shape[0]
    G = sim.make_bifurcation(seed=seed, strength=STRENGTH, n_per_time=n_per, dim=DIM)  # == the export
    s1 = dict(est_series=est, ours_series=[raw[0]] + est[1:], raw_series=raw,
              X0=np.asarray(z["X0"], np.float32),
              labels0=(np.asarray(z["labels0"]) if z["labels0"].size else None))
    cfg = json.loads(str(z["cfg"])) if "cfg" in z.files else {}
    EST[seed] = dict(G=G, s1=s1, cfg=cfg)
    bm8 = np.asarray(G.branch_means(8)); MID = float(bm8[:, 1].mean())
    bal = [float((G.project2d(c)[:, 1] > MID).mean()) for c in s1["ours_series"]]
    print(f"[seed {seed}] loaded {os.path.basename(_est_path(seed))} | n_per={n_per} "
          f"n_gen={int(z['n_gen'])} X0={s1['X0'].shape}")
    print(f"   est-cloud branch balance per time (0.5 = both branches): "
          + " ".join(f"t{t}={b:.2f}" for t, b in zip(TRAJ_T, bal)))
    print(f"   per-time W2 est|raw vs truth: "
          + " ".join(f"{w2(est[i], np.asarray(G.truth(t))):.2f}|{w2(raw[i], np.asarray(G.truth(t))):.2f}"
                     for i, t in enumerate(TRAJ_T)))
    if viz:
        from uotreg.plotting import show_estimates
        show_estimates(G, est, TRAJ_T, w2_fn=w2); plt.suptitle(f"seed {seed}: loaded estimate vs truth",
                                                               y=1.02); plt.show()
    return s1


def fit_seed(seed, K, viz=True, save=None):
    """Section-2: fit the trajectory methods on the LOADED estimate; score in 2-D AND full-d."""
    save = SAVE if save is None else save
    assert seed in EST, f"run load_seed({seed}) first"
    G, s1 = EST[seed]["G"], EST[seed]["s1"]
    series = [np.asarray(c, np.float32) for c in s1["ours_series"]]
    t0 = time.time()
    # project=False -> ambient-dim trajectories; we project ourselves, so ONE fit gives both metrics
    models = U.fit_trajectories(G, series, s1["raw_series"], None, TRAJ_T, K,
                                methods=METHODS, device=DEVICE, return_models=True, project=False)
    full = {m: np.asarray(models[m](s1["X0"]), np.float32) for m in METHODS}   # (T, N, DIM)
    trajs = {m: G.project_traj(f) for m, f in full.items()}                    # (T, N, 2)
    TRAJS[seed], FULL[seed] = trajs, full
    print(f"[seed {seed}] fit {len(METHODS)} methods ({time.time()-t0:.0f}s)")
    ROWS[seed] = TM.report(G, trajs, TRAJ_T, s1["labels0"], trajs_full=full)
    if save:
        os.makedirs(RESULTS_W, exist_ok=True)
        np.savez(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_trajs.npz"),
                 **{m.replace(' ', '_').replace(':', ''): np.asarray(t) for m, t in trajs.items()},
                 **{"full_" + m.replace(' ', '_').replace(':', ''): f for m, f in full.items()},
                 labels0=(s1["labels0"] if s1["labels0"] is not None else np.array([])),
                 K=json.dumps(dict(K)))
        with open(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_metrics.json"), "w") as f:
            json.dump({m: {k: (None if isinstance(v, float) and np.isnan(v) else float(v))
                           for k, v in r.items()} for m, r in ROWS[seed].items()}, f, indent=2)
    if viz:
        plot_trajectory_panel(G, trajs, max_cells=60, title=f"[bifurcation] d={DIM} seed {seed}"); plt.show()
    return ROWS[seed]


print(f"[traj-only bifurcation d={DIM}] SMOKE={SMOKE} device={DEVICE} TAG={TAG}")
print(f"  estimates <- {os.path.relpath(EST_ROOT, P.REPO)}  METHODS={METHODS}")
SEEDS      = [0] if SMOKE else list(range(10))   # SMOKE: one seed

## Section 1: load the saved estimates
Instant; each load prints a sanity check (per-time cloud balance + W2 of estimate vs raw).

In [ ]:
for k in SEEDS:
    load_seed(k)

## Section 2: trajectory fitting

In [ ]:
for k in SEEDS:
    fit_seed(k, dict(K_DEFAULT))

## Metrics: per-seed + aggregate

In [ ]:
seeds = sorted(ROWS)
print(f"[bifurcation d={DIM}] fitted seeds: {seeds}\n")
for s in seeds:
    print(f"--- seed {s} ---")
    print("   " + f"{'method':30s} " + " ".join(f"{h:>8}" for _, h in MK))
    for m in METHODS:
        print("   " + f"{m:30s} " + " ".join(
            ("     n/a" if (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k])) else f"{ROWS[s][m][k]:8.3f}")
            for k, _ in MK))
    print()

agg = {}
print(f"[bifurcation d={DIM}] AGGREGATE over {len(seeds)} seeds {seeds} (mean +/- std):")
print("   " + f"{'method':30s} " + " ".join(f"{h:>14}" for _, h in MK))
for m in METHODS:
    agg[m] = {}
    row = []
    for k, _ in MK:
        vals = [ROWS[s][m][k] for s in seeds
                if not (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k]))]
        mu, sd = (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), 0.0)
        agg[m][k] = {"mean": mu, "std": sd, "n": len(vals)}
        row.append(f"{mu:6.3f}+-{sd:<5.3f}")
    print("   " + f"{m:30s} " + " ".join(row))

if SAVE:
    os.makedirs(RESULTS_W, exist_ok=True)
    with open(os.path.join(RESULTS_W, f"{TAG}_agg_metrics.json"), "w") as f:
        json.dump({"seeds": seeds, "metrics": agg}, f, indent=2)
    print(f"\nsaved {TAG}_agg_metrics.json -> {os.path.relpath(RESULTS_W, P.REPO)}")